# » `Dependencias`:

In [1]:
# !pip install plotly
# !pip install sqlalchemy
# !pip install numpy
# !pip install matplotlib
%pip install Pillow
# %pip install io

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from sqlalchemy import create_engine
import os
from PIL import Image
from io import BytesIO

## 1. Conección → SIM(172.27.0.124)

In [3]:
# spring.datasource.url = jdbc:sqlserver://172.27.250.27;databaseName=SIRIM
SERVER = '172.27.0.124' # '172.27.0.242'
#DRIVER = 'SQL Server Native Client 11.0'
DRIVER = 'ODBC Driver 17 for SQL Server'
DATABASE = 'SIM'
USERNAME = 'userestadistica' # 'udesa'
PASSWORD = '$Us3R_3sT4d1sTic4$' # 'DESARROLLO2006'
DATABASE_CONNECTION = f'mssql://{USERNAME}:{PASSWORD}@{SERVER}/{DATABASE}?driver={DRIVER}'

engine = create_engine(DATABASE_CONNECTION)
connection = engine.connect()

## 2. Métodos genérico:

In [7]:
def get_query_sql(query):
  try:
    df = pd.read_sql(query, connection)
    return df
  except:
    print('¡Ocurrió un error!')

## 3. Lectura de ...

### 3.1 ...

In [8]:
# Extracción
id_persona = 'c59c57fe-e18a-4fb3-971b-166668127b9a'

SQL_QUERY = f'''

                  DECLARE @idPer UNIQUEIDENTIFIER = '{id_persona}'
                  IF EXISTS(SELECT TOP 1 1 FROM SimImagenExtranjero ie WHERE ie.uIdPersona = @idPer)
                  BEGIN
                     SELECT i.*
                     FROM (
                        SELECT
                           [sFileName] = CONCAT(p.sNombre, ', ', RTRIM(CONCAT(p.sPaterno, ' ', p.sMaterno))),
                           [sImageName] = CONCAT(p.sNombre, ', ', RTRIM(CONCAT(p.sPaterno, ' ', p.sMaterno)), '(', COALESCE(d.sNombre, ie.sTipo), ').jpg'),
                           [xImagen] = CAST(ie.xImagen AS VARBINARY(MAX))
                        FROM SIM.dbo.[SimImagenExtranjero] ie
                        LEFT JOIN SimDedo d ON ie.sIdDedo = d.sIdDedo
                        RIGHT JOIN SimPersona p ON ie.uIdPersona = p.uIdPersona
                        WHERE
                           ie.bUltimo = 1
                           AND p.uIdPersona = @idPer
                     ) i
                     ORDER BY
                        LEN(i.sFileName) DESC
                  END
                  ELSE
                  BEGIN
                     SELECT i.*
                     FROM (
                        SELECT 
                           [sFileName] = CONCAT(p.sNombre, ', ', RTRIM(CONCAT(p.sPaterno, ' ', p.sMaterno))),
                           [sImageName] = CONCAT(p.sNombre, ', ', RTRIM(CONCAT(p.sPaterno, ' ', p.sMaterno)), '(', COALESCE(d.strNombre, i.sTipoImagen), ').jpg'),
                           [xImagen] = CAST(i.xImagen AS VARBINARY(MAX))
                        FROM SIM.dbo.[SimImagen] i
                        LEFT JOIN SimDedoBio d ON i.sTipoImagen = d.strDedoBio
                        RIGHT JOIN SimPersona p ON i.uIdPersona = p.uIdPersona
                        WHERE
                           i.bUltimo = 1
                           AND p.uIdPersona = @idPer
                     ) i
                     ORDER BY
                        LEN(i.sFileName) DESC
                  END


'''

df_per_bio = get_query_sql(SQL_QUERY)

In [9]:
# Crea carpeta si no existe
root_directory = 'sim_bio'
if not os.path.exists(root_directory):
    os.makedirs(root_directory)

# Convierte campo VARBINARY a imagen y deposita en carpeta
for i, file in df_per_bio.iterrows():
   file_name = file['sFileName']
   dir_of_bio = os.path.join(root_directory, file_name)

   # Crear la carpeta
   if not os.path.exists(dir_of_bio):
      os.makedirs(dir_of_bio)

   # imagenes
   df_imgs_of_curr_file = df_per_bio.loc[df_per_bio['sFileName'] == file_name]

   for ii, img in df_imgs_of_curr_file.iterrows():
      img_name = img['sImageName']
      img_bytes = img['xImagen']
      imagen = Image.open(BytesIO(img_bytes))
      imagen_rgb = imagen.convert('RGB')
      imagen_rgb.save(os.path.join(dir_of_bio, img_name))